# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shamiquekhan/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup: navigate to repo root
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shamiquekhan/flyrank-ml-internship", "flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
print("Working dir:", os.getcwd())

Working dir: /home/shamique/flyrank/ml1/flyrank-ml-internship


## 1. My lane as an ML task (type)

I'm working on **Refresh / Content Opportunity Scoring** (Lane 2). I frame this as a **binary classification** problem: given observable page-level signals, I predict whether a page is currently declining in search performance. The output — a probability of decline — directly translates into a priority score that reviewers use to rank pages.

I chose classification over regression or ranking for two reasons. First, the starter dataset provides a clean label (`trend_direction == "down"`) that I can use as my target. Second, a classification model's probability output naturally produces a ranked queue, which matches the business goal: reviewers work through pages in order of predicted decline risk.

## 2. Target or proxy

My target is `is_declining_label`: a binary flag where 1 means the page's trend direction is "down" and 0 means any other direction (stable, flat, up, or new).

This is a **proxy label** — it measures current trend direction, not a future outcome. A stronger capstone would define a future-looking target (e.g. "decline over the next 30 days using features from the prior 90 days"), but this proxy is good enough for a first model. It comes from the `trend_direction` column in the starter dataset, which is derived from observed search data — not from any product decision flag.

## 3. Success metric

**Primary metric: Precision@50.**

This measures: of the top 50 pages my model flags for review, what fraction are actually declining? I chose this because it directly matches the business constraint — a content team has limited capacity and can realistically review only about 50 pages at a time. Precision@50 tells me how many of those 50 reviews would be well-spent.

**Secondary metric: ROC-AUC.**

This measures the model's overall ability to separate declining from non-declining pages across all thresholds. It gives me a fuller picture of model quality beyond just the top 50.

## 4. The unit of analysis, as a real dataframe

*I load the starter dataset and look at a few rows — one row = one webpage.*

In [2]:
# Load the starter dataset
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 3 rows show the unit of analysis:")
df.head(3)

Shape: 30000 rows, 44 columns

First 3 rows show the unit of analysis:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


One row = **one webpage (content_id)**. Each column is a feature describing that page: its search volume, impressions, clicks, CTR, average position, content age, word count, engagement metrics, and the current trend direction. The full data dictionary in `docs/data-dictionary.md` explains all 44 columns.

Now I sketch what my target column looks like:

In [3]:
# Sketch the target column
df["target"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("Target distribution:")
print(df["target"].value_counts().to_string())
print()

print("Sample rows with target:")
df[["content_id", "trend_direction", "target", "impressions_90d", "avg_position"]].head(10)

Target distribution:
target
1    16262
0    13738

Sample rows with target:


,content_id,trend_direction,target,impressions_90d,avg_position
0,content_304f48230142,down,1,3803,10.6
1,content_a1fb4e703a9e,down,1,15320,20.3
2,content_9aa793d4d895,down,1,12581,36.5
3,content_331d6c4de07b,stable,0,11751,6.2
4,content_d99b7a2d90ca,down,1,19140,44.0
5,content_d4084a4bc775,down,1,3970,8.5
6,content_9a34b442b552,down,1,20,7.0
7,content_a63219c6e95a,stable,0,1724,21.2
8,content_5e6c160719bc,down,1,32574,46.0
9,content_c27558df2b0c,down,1,1240,4.9


The target is reasonably balanced: ~54% of pages are classified as declining (target=1), ~46% as not declining (target=0). This balance means accuracy alone won't be misleading — but I still prefer Precision@50 because it reflects how the output is actually used.

## 5. Why ML beats a fixed rule here

A fixed rule can capture simple patterns — like "if a page hasn't been updated in 180 days and still gets traffic, flag it." But search performance depends on many interacting factors: content age, position, CTR, competition, engagement, and seasonality all interact in ways a hand-coded if-statement cannot capture cleanly.

For example, a newer page at position 5 with high CTR might not need refresh, while an older page at position 2 with declining CTR might. The relationship between these signals shifts over time as search patterns change. Machine learning learns these interactions from historical data and adapts as new data arrives, producing a more accurate ranking than any static rule I could write by hand.

The starter pipeline already demonstrates this: the hand-written baseline achieves Precision@50 of 0.240, while the random forest reaches 0.680 — a ~2.8x improvement. That measured gap is my evidence that ML adds value over rules for this problem.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.